###E-Commerce Cohort Analysis Pipeline
Transforming raw e-commerce data into a cohort analysis model using Databricks.

####01 - Data Ingestion

In [0]:
-- Ingest data source / create schema

CREATE SCHEMA IF NOT EXISTS ubiratan_cohort_project;
USE ubiratan_cohort_project;

#### 02 - Bronze layer
**Purpose:** Standardize column types and trim whitespace for the raw archive.

In [0]:
-- Create Bronze Delta Table

CREATE OR REPLACE TABLE delta.`/Volumes/workspace/ubiratan_cohort_project/source_files/delta_source_data`
USING DELTA
AS
SELECT *
FROM csv.`/Volumes/workspace/ubiratan_cohort_project/source_files/*`;

num_affected_rows,num_inserted_rows


In [0]:
-- Inspect Bronze Delta Table

SELECT *
FROM delta.`/Volumes/workspace/ubiratan_cohort_project/source_files/delta_source_data`
LIMIT 5;

_c0,_c1
PAR1 �>�>Ǽ�� �  6 (�        � ��   ��     �   �   7     6     �  ~     D  �  �   L  \  [   �  �     v   o  e  �  �   �  �  �     �    �     9      m  �  �   �   �  �   #   �  2  p  T   [  s     �  �   �   R  �   �  �  (   /  �  W       �   �  �  v  V  �   N  �  ,"   �   (  `      �  �   w   &   �  �  j  >  �  �  �  �  F      )  �  �  g  �  N  U    ?  �  �   �  �   �  �  �  �  Y  �  U   �    _       �    �   �   �       �  P  Z  d     �  �  Q  �  �  x    =   �  &  �  e  �  �   �  �     �  �  5      �  �    �  7  ^  q   �   �  �   �  �   :  r  �  �  (  }  k  �   �  ""      ;    �   �     |  Y   �     �  t  1   |  �  �  =  M  �   �   _  t      �  )   s  O  �  �     T  �  9  j   �  n  E  �  n     q  F  ;    +    L  "
  �  �  6  o  !  �  �   �  c  *  A  k  �  y  �   R  >  )  ^  �   �  r  �   q  �  �  �  �   �     :  \  �        j  R      �  J   �  B  _  �  �  �   �  �  �  �  �  �  �     �  �  �  .  �     9     K  �  �  �  W  �  �  �  �  �   E  �  �   �  �  �  �  �   |  �  �   D   �  �   `  �   H    �   �  �  �   �  �  �  z  S   �  �    T  �   �  �  �     �  <  �     7  ~   �     �   O  �  �  �   S  �  �  �  �  �  V  �  �  �  x    (  �  �        J  �  �  d  �  �  !  �    2    i  ,null
"  �  �  �   �  ]   �   �  8     �  �   �  $     0  �  �  �  �   �  �      �  �  �  �   h  �  O  o  �  ;   �  N  d  )  =  A   �    l  �  \   �  �  h   6   E   C  �  �   �  �  �  �  f  B  �  �  ?   ""  S  ~  �  �  �  Z  8   ""   ]  4   �  �  �   $  �  �  m   %  �   5  w  �   �        j  �   �      �   �  h      .   K  &  �  �  �   }  �  �  4  �   �    6  y  �  �     �  i       b  �  �  X  �  @     �   �  X   u   �  �  �  a  �   5  �  .  ?    �  �  Q   l   �   �  �  �  �   �  M  %   �  �   �   �   1  2        3  �  G  �  Y  *   l  g   G    �  0  �   �  L   �    �   {  �    �  T  ;  [  z  }   �  �   �   i  2   �  e  L  H     �  s  �     $  '  S  �  ",null
  �  �  �   3  �  #  \  A  v  �  G      �   7       1  C  W  �  �   �  C      �    �   %  �   �   `  .  M  �  �  �  �  �   Y  �  +    %  K  b  -   >   �  u  �  J    �  �   �   #  �   *  �  �   �  u  M   �    �  �  R  y   �   �   �       e   E  �  �  �  <  f  �  I   �  Q  �   �  ]  �   �   }  �  �   �  U  �  _  �  �     �  I  ^   �    �   �    '  �  �  �  -  �     8  �  c     H  :     $   B  �     g  V  /   �     �  h  �    �  �      *    �  �  x  �   {  ~  �  O   m  �   �  1    ?  �  w  |   +  4  �  ,null
  �  �  �    �  ,  �  


#### 03 - Silver layer (cleaned & deduplicated) 
**Purpose:** Deduplicate by order_id and remove invalid records.

In [0]:
--  Create Silver Delta Table (Rename Columns & Filter Header)

CREATE OR REPLACE TABLE clean_orders
USING DELTA
AS
WITH deduplicated_data AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY _c0 ORDER BY _c0 ASC) AS row_num
    FROM delta.`/Volumes/workspace/ubiratan_cohort_project/source_files/delta_source_data`
    WHERE _c0 IS NOT NULL
)
SELECT 
    _c0 AS row_id
FROM deduplicated_data
WHERE row_num = 1;

num_affected_rows,num_inserted_rows


In [0]:
-- Validate Silver Delta Table

SELECT 
  COUNT(*) AS total_rows,
  COUNT(DISTINCT order_id) AS unique_orders,
  MIN(order_date) AS earliest_order,
  MAX(order_date) AS latest_order,
  SUM(sales) AS total_sales
FROM workspace.default.clean_orders;

total_rows,unique_orders,earliest_order,latest_order,total_sales
1000,1000,2024-01-01,2024-12-30,158409.76


####04 - Gold layer (cohort modeling)
**Purpose:** Create a customer-level table with first and second purchase dates.

In [0]:
-- Create Gold Delta Table (Cohort Modeling)

CREATE TABLE IF NOT EXISTS cohort_customers
USING DELTA
AS
WITH first_purchase AS (
    -- Step 1: Find the first purchase date for each customer
    SELECT
        customer_id,
        MIN(order_date) AS cohort_date,
        DATE_FORMAT(MIN(order_date), 'yyyy-MM') AS cohort_month
    FROM workspace.default.clean_orders
    GROUP BY customer_id
),
second_purchase AS (
    -- Step 2: Find the second purchase date for each customer
    SELECT
        customer_id,
        MIN(order_date) AS second_purchase_date
    FROM workspace.default.clean_orders
    WHERE order_date > (
        SELECT MIN(order_date) 
        FROM workspace.default.clean_orders c2 
        WHERE c2.customer_id = workspace.default.clean_orders.customer_id
    )
    GROUP BY customer_id
)
SELECT
    f.customer_id,
    f.cohort_date,
    f.cohort_month,
    s.second_purchase_date,
    DATEDIFF(s.second_purchase_date, f.cohort_date) AS days_to_second_purchase
FROM first_purchase f
LEFT JOIN second_purchase s
    ON f.customer_id = s.customer_id;

num_affected_rows,num_inserted_rows


In [0]:
-- Validate Gold Delta Table

SELECT
    COUNT(*) AS total_customers,
    COUNT(second_purchase_date) AS repeat_customers,
    COUNT(*) - COUNT(second_purchase_date) AS one_time_customers,
    ROUND(COUNT(second_purchase_date) * 100.0 / COUNT(*), 2) AS repeat_rate_pct
FROM cohort_customers;

total_customers,repeat_customers,one_time_customers,repeat_rate_pct
888,580,308,65.32


#### 05 - Reporting Queries & Metrics


In [0]:
-- Metrics 01- Cohort Size (New Customers)

SELECT
    cohort_month,
    COUNT(customer_id) AS total_customers
FROM ubiratan_cohort_project.cohort_customers
GROUP BY cohort_month
ORDER BY cohort_month;

cohort_month,total_customers
2024-01-01T00:00:00.000Z,78
2024-02-01T00:00:00.000Z,79
2024-03-01T00:00:00.000Z,72
2024-04-01T00:00:00.000Z,53
2024-05-01T00:00:00.000Z,49
2024-06-01T00:00:00.000Z,56
2024-07-01T00:00:00.000Z,54
2024-08-01T00:00:00.000Z,47
2024-09-01T00:00:00.000Z,31
2024-10-01T00:00:00.000Z,45


In [0]:
-- Metrics 02 - Retention to 2nd Purchase

SELECT
    cohort_month,
    COUNT(customer_id) AS total_customers,
    COUNT(second_purchase_date) AS repeat_customers,
    ROUND(COUNT(second_purchase_date) * 100.0 / COUNT(customer_id), 2) AS retention_pct,
    ROUND(AVG(days_to_second_purchase), 1) AS avg_days_to_second_purchase
FROM ubiratan_cohort_project.cohort_customers
GROUP BY cohort_month
ORDER BY cohort_month;

cohort_month,total_customers,repeat_customers,retention_pct,avg_days_to_second_purchase
2024-01-01T00:00:00.000Z,78,60,76.92,323.7
2024-02-01T00:00:00.000Z,79,62,78.48,258.1
2024-03-01T00:00:00.000Z,72,57,79.17,259.8
2024-04-01T00:00:00.000Z,53,33,62.26,158.5
2024-05-01T00:00:00.000Z,49,33,67.35,222.0
2024-06-01T00:00:00.000Z,56,42,75.00,305.4
2024-07-01T00:00:00.000Z,54,29,53.70,208.2
2024-08-01T00:00:00.000Z,47,26,55.32,269.7
2024-09-01T00:00:00.000Z,31,19,61.29,247.3
2024-10-01T00:00:00.000Z,45,24,53.33,206.0


In [0]:
-- Metrics 03 - Repeat Purchase Depth

WITH purchase_counts AS (
    SELECT
        customer_id,
        COUNT(order_id) AS total_purchases
    FROM workspace.default.clean_orders
    GROUP BY customer_id
)
SELECT
    c.cohort_month,
    COUNT(p.customer_id) AS total_customers,
    ROUND(SUM(CASE WHEN p.total_purchases >= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(p.customer_id), 2) AS pct_2_plus,
    ROUND(SUM(CASE WHEN p.total_purchases >= 3 THEN 1 ELSE 0 END) * 100.0 / COUNT(p.customer_id), 2) AS pct_3_plus,
    ROUND(SUM(CASE WHEN p.total_purchases >= 4 THEN 1 ELSE 0 END) * 100.0 / COUNT(p.customer_id), 2) AS pct_4_plus
FROM ubiratan_cohort_project.cohort_customers c
JOIN purchase_counts p
  ON c.customer_id = p.customer_id
GROUP BY c.cohort_month
ORDER BY c.cohort_month;

cohort_month,total_customers,pct_2_plus,pct_3_plus,pct_4_plus
2024-01-01T00:00:00.000Z,78,37.18,25.64,11.54
2024-02-01T00:00:00.000Z,79,49.37,30.38,11.39
2024-03-01T00:00:00.000Z,72,41.67,26.39,12.50
2024-04-01T00:00:00.000Z,53,47.17,35.85,11.32
2024-05-01T00:00:00.000Z,49,34.69,24.49,10.20
2024-06-01T00:00:00.000Z,56,23.21,5.36,3.57
2024-07-01T00:00:00.000Z,54,22.22,11.11,3.70
2024-08-01T00:00:00.000Z,47,10.64,2.13,0.00
2024-09-01T00:00:00.000Z,31,12.90,3.23,3.23
2024-10-01T00:00:00.000Z,45,11.11,2.22,0.00


%md
####06 - Adding New Data (2025)

In [0]:
-- Append 2025 into Bronze Delta Table
-- Insert 2025 data directly into workspace.default.clean_orders 

INSERT INTO workspace.default.clean_orders
WITH raw_data AS (
    SELECT 
        CAST(row_id AS INT) AS row_id,
        customer_id,
        CAST(order_date AS DATE) AS order_date,
        order_id,
        CAST(sales AS DECIMAL(10,2)) AS sales
    FROM read_files('/Volumes/workspace/ubiratan_cohort_project/source_files/ecom_orders_incremental_2025_rebalanced.csv', format => 'csv', header => true)
    WHERE row_id IS NOT NULL
      AND customer_id IS NOT NULL
      AND order_id IS NOT NULL
      AND TRY_CAST(row_id AS INT) IS NOT NULL
      AND TRY_CAST(sales AS DECIMAL(10,2)) > 0
),
deduplicated_data AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY row_id ASC) AS row_num
    FROM raw_data
)
SELECT 
    row_id,
    customer_id,
    order_date,
    order_id,
    sales
FROM deduplicated_data
WHERE row_num = 1;

num_affected_rows,num_inserted_rows
1000,1000


In [0]:
-- Rebuild Silver Delta Table

CREATE OR REPLACE TABLE clean_orders
USING DELTA
AS
WITH raw_data AS (
    SELECT 
        row_id,
        customer_id,
        order_date,
        order_id,
        sales
    FROM read_files('/Volumes/workspace/ubiratan_cohort_project/source_files/*.csv', format => 'csv', header => true)
    WHERE row_id IS NOT NULL
      AND customer_id IS NOT NULL
      AND order_id IS NOT NULL
      AND TRY_CAST(row_id AS INT) IS NOT NULL
      AND TRY_CAST(sales AS DECIMAL(10,2)) > 0
),
deduplicated_data AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY CAST(row_id AS INT) ASC) AS row_num
    FROM raw_data
)
SELECT 
    CAST(row_id AS INT)            AS row_id,
    customer_id                    AS customer_id,
    CAST(order_date AS DATE)       AS order_date,
    order_id                       AS order_id,
    CAST(sales AS DECIMAL(10,2))   AS sales
FROM deduplicated_data
WHERE row_num = 1;

num_affected_rows,num_inserted_rows


In [0]:
-- Rebuild Gold Delta Table

CREATE TABLE IF NOT EXISTS workspace.ubiratan_cohort_project.cohort_customers
USING DELTA
AS
WITH base AS (
  SELECT
    customer_id,
    CAST(order_date AS DATE) AS order_date
  FROM workspace.ubiratan_cohort_project.clean_orders
  WHERE customer_id IS NOT NULL
    AND order_date IS NOT NULL
),
first_purchase AS (
  SELECT
    customer_id,
    MIN(order_date) AS cohort_date,
    DATE_FORMAT(MIN(order_date), 'yyyy-MM') AS cohort_month
  FROM base
  GROUP BY customer_id
),
second_purchase AS (
  SELECT
    customer_id,
    MIN(order_date) AS second_purchase_date
  FROM base
  WHERE order_date > (
    SELECT MIN(order_date)
    FROM base b2
    WHERE b2.customer_id = base.customer_id
  )
  GROUP BY customer_id
)
SELECT
  f.customer_id,
  f.cohort_date,
  f.cohort_month,
  s.second_purchase_date,
  DATEDIFF(s.second_purchase_date, f.cohort_date) AS days_to_second_purchase
FROM first_purchase f
LEFT JOIN second_purchase s
  ON f.customer_id = s.customer_id;



num_affected_rows,num_inserted_rows


In [0]:
-- Verify Gold Delta Table

WITH stats AS (
  SELECT
    COUNT(*) AS total_customers,
    COUNT(second_purchase_date) AS repeat_customers,
    COUNT(*) - COUNT(second_purchase_date) AS one_time_customers,

    SUM(CASE WHEN first_purchase_date IS NULL THEN 1 ELSE 0 END) AS cohort_date_nulls,
    SUM(CASE WHEN second_purchase_date IS NULL THEN 1 ELSE 0 END) AS second_purchase_nulls,

    MIN(first_purchase_date) AS min_cohort_date,
    MAX(first_purchase_date) AS max_cohort_date,

    MIN(cohort_month) AS min_cohort_month,
    MAX(cohort_month) AS max_cohort_month
  FROM workspace.ubiratan_cohort_project.cohort_customers
),
coverage_2025 AS (
  SELECT
    COUNT(*) AS customers_with_cohort_date_in_2025
  FROM workspace.ubiratan_cohort_project.cohort_customers
  WHERE CAST(first_purchase_date AS DATE) >= DATE '2025-01-01'
    AND CAST(first_purchase_date AS DATE) <  DATE '2026-01-01'
)
SELECT
  s.total_customers,
  s.repeat_customers,
  s.one_time_customers,
  ROUND(100.0 * s.repeat_customers / NULLIF(s.total_customers, 0), 2) AS repeat_rate_pct,

  s.cohort_date_nulls,
  s.second_purchase_nulls,

  s.min_cohort_date,
  s.max_cohort_date,
  s.min_cohort_month,
  s.max_cohort_month,

  c.customers_with_cohort_date_in_2025
FROM stats s
CROSS JOIN coverage_2025 c;

total_customers,repeat_customers,one_time_customers,repeat_rate_pct,cohort_date_nulls,second_purchase_nulls,min_cohort_date,max_cohort_date,min_cohort_month,max_cohort_month,customers_with_cohort_date_in_2025
888,580,308,65.32,0,308,2024-01-01,2025-12-31,2024-01-01T00:00:00.000Z,2025-12-01T00:00:00.000Z,239


#### 07 - Business Insights

In [0]:
-- 2025 Cohort Retention Analysis

SELECT 
    cohort_month,
    COUNT(*) AS cohort_size,
    COUNT(second_purchase_date) AS repeat_customers,
    ROUND(COUNT(second_purchase_date) * 100.0 / COUNT(*), 2) AS repeat_rate_pct,

    ROUND(
      COUNT(CASE 
              WHEN second_purchase_date IS NOT NULL
               AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 1
              THEN 1 
            END) * 100.0 / COUNT(*), 2
    ) AS retention_1m,

    ROUND(
      COUNT(CASE 
              WHEN second_purchase_date IS NOT NULL
               AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 2
              THEN 1 
            END) * 100.0 / COUNT(*), 2
    ) AS retention_2m,

    ROUND(
      COUNT(CASE 
              WHEN second_purchase_date IS NOT NULL
               AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 3
              THEN 1 
            END) * 100.0 / COUNT(*), 2
    ) AS retention_3m,

    ROUND(
      AVG(CASE WHEN second_purchase_date IS NOT NULL THEN days_to_second_purchase END),
      1
    ) AS avg_days_to_2nd
FROM workspace.ubiratan_cohort_project.cohort_customers
WHERE YEAR(cohort_month) = 2025
GROUP BY cohort_month
ORDER BY cohort_month;

cohort_month,cohort_size,repeat_customers,repeat_rate_pct,retention_1m,retention_2m,retention_3m,avg_days_to_2nd
2025-01-01T00:00:00.000Z,43,35,81.40,6.98,18.60,32.56,148.7
2025-02-01T00:00:00.000Z,32,25,78.13,15.63,21.88,31.25,112.7
2025-03-01T00:00:00.000Z,17,11,64.71,23.53,35.29,35.29,103.0
2025-04-01T00:00:00.000Z,27,20,74.07,25.93,29.63,40.74,90.7
2025-05-01T00:00:00.000Z,25,18,72.00,8.00,20.00,36.00,104.9
2025-06-01T00:00:00.000Z,19,16,84.21,21.05,42.11,42.11,80.1
2025-07-01T00:00:00.000Z,22,14,63.64,9.09,22.73,36.36,93.2
2025-08-01T00:00:00.000Z,9,5,55.56,22.22,22.22,22.22,66.4
2025-09-01T00:00:00.000Z,13,5,38.46,15.38,23.08,30.77,45.2
2025-10-01T00:00:00.000Z,11,4,36.36,27.27,36.36,36.36,22.0


In [0]:
-- Compare 2024 vs 2025 Cohorts

SELECT 
    YEAR(cohort_month) AS cohort_year,
    COUNT(*) AS total_customers,
    COUNT(second_purchase_date) AS repeat_customers,
    ROUND(COUNT(second_purchase_date) * 100.0 / COUNT(*), 2) AS repeat_rate_pct,
    
    ROUND(COUNT(CASE 
            WHEN second_purchase_date IS NOT NULL
             AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 1
            THEN 1 
         END) * 100.0 / COUNT(*), 2) AS retention_1m,

    ROUND(COUNT(CASE 
            WHEN second_purchase_date IS NOT NULL
             AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 2
            THEN 1 
         END) * 100.0 / COUNT(*), 2) AS retention_2m,

    ROUND(COUNT(CASE 
            WHEN second_purchase_date IS NOT NULL
             AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 3
            THEN 1 
         END) * 100.0 / COUNT(*), 2) AS retention_3m,
    
    ROUND(AVG(CASE WHEN second_purchase_date IS NOT NULL THEN days_to_second_purchase END), 1) AS avg_days_to_2nd,
    ROUND(MIN(CASE WHEN second_purchase_date IS NOT NULL THEN days_to_second_purchase END), 0) AS fastest_repurchase,
    ROUND(MAX(CASE WHEN second_purchase_date IS NOT NULL THEN days_to_second_purchase END), 0) AS slowest_repurchase
FROM workspace.ubiratan_cohort_project.cohort_customers
WHERE YEAR(cohort_month) IN (2024, 2025)
GROUP BY YEAR(cohort_month)
ORDER BY cohort_year;

cohort_year,total_customers,repeat_customers,repeat_rate_pct,retention_1m,retention_2m,retention_3m,avg_days_to_2nd,fastest_repurchase,slowest_repurchase
2024,649,427,65.79,6.47,12.63,17.26,247.5,1,712
2025,239,153,64.02,14.23,23.43,31.80,105.2,2,333


In [0]:
-- Retention Curves by Cohort Month

WITH retention_by_month AS (
    SELECT 
        date_format(cohort_month, 'yyyy-MM') AS cohort_month,
        COUNT(*) AS cohort_size,
        
        100.0 AS month_0,
        
        ROUND(COUNT(CASE 
                WHEN second_purchase_date IS NOT NULL
                 AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 1
                THEN 1 
             END) * 100.0 / COUNT(*), 2) AS month_1,

        ROUND(COUNT(CASE 
                WHEN second_purchase_date IS NOT NULL
                 AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 2
                THEN 1 
             END) * 100.0 / COUNT(*), 2) AS month_2,

        ROUND(COUNT(CASE 
                WHEN second_purchase_date IS NOT NULL
                 AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 3
                THEN 1 
             END) * 100.0 / COUNT(*), 2) AS month_3,

        ROUND(COUNT(CASE 
                WHEN second_purchase_date IS NOT NULL
                 AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 4
                THEN 1 
             END) * 100.0 / COUNT(*), 2) AS month_4,

        ROUND(COUNT(CASE 
                WHEN second_purchase_date IS NOT NULL
                 AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 5
                THEN 1 
             END) * 100.0 / COUNT(*), 2) AS month_5,

        ROUND(COUNT(CASE 
                WHEN second_purchase_date IS NOT NULL
                 AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 6
                THEN 1 
             END) * 100.0 / COUNT(*), 2) AS month_6
        
    FROM workspace.ubiratan_cohort_project.cohort_customers
    GROUP BY date_format(cohort_month, 'yyyy-MM')
)
SELECT 
    cohort_month,
    cohort_size,
    month_0, month_1, month_2, month_3, month_4, month_5, month_6
FROM retention_by_month
ORDER BY cohort_month DESC
LIMIT 15;

cohort_month,cohort_size,month_0,month_1,month_2,month_3,month_4,month_5,month_6
2025-12,14,100.0,0.00,0.00,0.00,0.00,0.00,0.00
2025-11,7,100.0,0.00,0.00,0.00,0.00,0.00,0.00
2025-10,11,100.0,27.27,36.36,36.36,36.36,36.36,36.36
2025-09,13,100.0,15.38,23.08,30.77,38.46,38.46,38.46
2025-08,9,100.0,22.22,22.22,22.22,55.56,55.56,55.56
2025-07,22,100.0,9.09,22.73,36.36,40.91,50.00,63.64
2025-06,19,100.0,21.05,42.11,42.11,63.16,73.68,78.95
2025-05,25,100.0,8.00,20.00,36.00,44.00,56.00,64.00
2025-04,27,100.0,25.93,29.63,40.74,44.44,55.56,66.67
2025-03,17,100.0,23.53,35.29,35.29,41.18,47.06,47.06


In [0]:
-- Seasonal Patterns Analysis

SELECT 
    MONTH(cohort_month) AS calendar_month,
    CASE MONTH(cohort_month)
        WHEN 1 THEN 'January'
        WHEN 2 THEN 'February'
        WHEN 3 THEN 'March'
        WHEN 4 THEN 'April'
        WHEN 5 THEN 'May'
        WHEN 6 THEN 'June'
        WHEN 7 THEN 'July'
        WHEN 8 THEN 'August'
        WHEN 9 THEN 'September'
        WHEN 10 THEN 'October'
        WHEN 11 THEN 'November'
        WHEN 12 THEN 'December'
    END AS month_name,

    -- Overall metrics
    COUNT(*) AS total_customers,
    ROUND(COUNT(second_purchase_date) * 100.0 / COUNT(*), 2) AS repeat_rate_pct,

    -- Retention metrics
    ROUND(
      COUNT(CASE 
              WHEN second_purchase_date IS NOT NULL
               AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 1
              THEN 1 
            END) * 100.0 / COUNT(*), 2
    ) AS retention_1m,

    ROUND(
      COUNT(CASE 
              WHEN second_purchase_date IS NOT NULL
               AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 3
              THEN 1 
            END) * 100.0 / COUNT(*), 2
    ) AS retention_3m,

    ROUND(AVG(CASE WHEN second_purchase_date IS NOT NULL THEN days_to_second_purchase END), 1) AS avg_days_to_2nd
FROM workspace.ubiratan_cohort_project.cohort_customers
GROUP BY MONTH(cohort_month)
ORDER BY calendar_month;

calendar_month,month_name,total_customers,repeat_rate_pct,retention_1m,retention_3m,avg_days_to_2nd
1,January,121,78.51,5.79,23.14,259.2
2,February,111,78.38,10.81,24.32,216.3
3,March,89,76.40,11.24,24.72,234.4
4,April,80,66.25,18.75,33.75,132.9
5,May,74,68.92,8.11,25.68,180.7
6,June,75,77.33,8.00,14.67,243.3
7,July,76,56.58,7.89,22.37,170.7
8,August,56,55.36,3.57,8.93,236.9
9,September,44,54.55,6.82,15.91,205.2
10,October,56,50.00,10.71,17.86,179.7


In [0]:
-- Save Analysis Table

-- 1) Save: 2024 vs 2025 (Year-over-Year) comparison
CREATE OR REPLACE TABLE workspace.ubiratan_cohort_project.analysis_year_comparison AS
SELECT 
    YEAR(cohort_month) AS cohort_year,
    COUNT(*) AS total_customers,
    COUNT(second_purchase_date) AS repeat_customers,
    ROUND(COUNT(second_purchase_date) * 100.0 / COUNT(*), 2) AS repeat_rate_pct,
    ROUND(COUNT(CASE WHEN MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 1 THEN 1 END) * 100.0 / COUNT(*), 2) AS retention_1m,
    ROUND(COUNT(CASE WHEN MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 2 THEN 1 END) * 100.0 / COUNT(*), 2) AS retention_2m,
    ROUND(COUNT(CASE WHEN MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 3 THEN 1 END) * 100.0 / COUNT(*), 2) AS retention_3m,
    ROUND(AVG(CASE WHEN second_purchase_date IS NOT NULL THEN days_to_second_purchase END), 1) AS avg_days_to_2nd,
    ROUND(MIN(CASE WHEN second_purchase_date IS NOT NULL THEN days_to_second_purchase END), 0) AS fastest_repurchase,
    ROUND(MAX(CASE WHEN second_purchase_date IS NOT NULL THEN days_to_second_purchase END), 0) AS slowest_repurchase
FROM workspace.ubiratan_cohort_project.cohort_customers
GROUP BY YEAR(cohort_month);

-- 2) Save: Retention Curves by Cohort Month (month-only)
CREATE OR REPLACE TABLE workspace.ubiratan_cohort_project.analysis_retention_curves AS
WITH retention_by_month AS (
    SELECT 
        date_format(cohort_month, 'yyyy-MM') AS cohort_month,
        COUNT(*) AS cohort_size,
        100.0 AS month_0,
        ROUND(COUNT(CASE 
                WHEN second_purchase_date IS NOT NULL
                 AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 1
                THEN 1 END) * 100.0 / COUNT(*), 2) AS month_1,
        ROUND(COUNT(CASE 
                WHEN second_purchase_date IS NOT NULL
                 AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 2
                THEN 1 END) * 100.0 / COUNT(*), 2) AS month_2,
        ROUND(COUNT(CASE 
                WHEN second_purchase_date IS NOT NULL
                 AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 3
                THEN 1 END) * 100.0 / COUNT(*), 2) AS month_3,
        ROUND(COUNT(CASE 
                WHEN second_purchase_date IS NOT NULL
                 AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 4
                THEN 1 END) * 100.0 / COUNT(*), 2) AS month_4,
        ROUND(COUNT(CASE 
                WHEN second_purchase_date IS NOT NULL
                 AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 5
                THEN 1 END) * 100.0 / COUNT(*), 2) AS month_5,
        ROUND(COUNT(CASE 
                WHEN second_purchase_date IS NOT NULL
                 AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 6
                THEN 1 END) * 100.0 / COUNT(*), 2) AS month_6
    FROM workspace.ubiratan_cohort_project.cohort_customers
    GROUP BY date_format(cohort_month, 'yyyy-MM')
)
SELECT 
    cohort_month,
    cohort_size,
    month_0, month_1, month_2, month_3, month_4, month_5, month_6
FROM retention_by_month;

-- 3) Save: Seasonal Patterns (month-only, no quarter split)
CREATE OR REPLACE TABLE workspace.ubiratan_cohort_project.analysis_seasonal_patterns AS
SELECT 
    MONTH(cohort_month) AS calendar_month,
    CASE MONTH(cohort_month)
        WHEN 1 THEN 'January'
        WHEN 2 THEN 'February'
        WHEN 3 THEN 'March'
        WHEN 4 THEN 'April'
        WHEN 5 THEN 'May'
        WHEN 6 THEN 'June'
        WHEN 7 THEN 'July'
        WHEN 8 THEN 'August'
        WHEN 9 THEN 'September'
        WHEN 10 THEN 'October'
        WHEN 11 THEN 'November'
        WHEN 12 THEN 'December'
    END AS month_name,
    COUNT(*) AS total_customers,
    ROUND(COUNT(second_purchase_date) * 100.0 / COUNT(*), 2) AS repeat_rate_pct,
    ROUND(COUNT(CASE 
            WHEN second_purchase_date IS NOT NULL
             AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 1 THEN 1 END) * 100.0 / COUNT(*), 2) AS retention_1m,
    ROUND(COUNT(CASE 
            WHEN second_purchase_date IS NOT NULL
             AND MONTHS_BETWEEN(second_purchase_date, first_purchase_date) <= 3 THEN 1 END) * 100.0 / COUNT(*), 2) AS retention_3m,
    ROUND(AVG(CASE WHEN second_purchase_date IS NOT NULL THEN days_to_second_purchase END), 1) AS avg_days_to_2nd
FROM workspace.ubiratan_cohort_project.cohort_customers
GROUP BY MONTH(cohort_month);

-- Verify
SHOW TABLES IN workspace.ubiratan_cohort_project LIKE 'analysis_%';

database,tableName,isTemporary


#### Visualizations
The two charts below demonstrate the total cohort customer base

In [0]:
-- Bar Chart - Cohort Size (2024 only)

SELECT
  date_trunc('month', cohort_month) AS cohort_month,
  COUNT(DISTINCT customer_id) AS customer_id
FROM workspace.ubiratan_cohort_project.cohort_customers
WHERE cohort_month >= DATE '2024-01-01'
  AND cohort_month <  DATE '2025-01-01'
GROUP BY date_trunc('month', cohort_month)
ORDER BY cohort_month;

cohort_month,customer_id
2024-01-01T00:00:00.000Z,78
2024-02-01T00:00:00.000Z,79
2024-03-01T00:00:00.000Z,72
2024-04-01T00:00:00.000Z,53
2024-05-01T00:00:00.000Z,49
2024-06-01T00:00:00.000Z,56
2024-07-01T00:00:00.000Z,54
2024-08-01T00:00:00.000Z,47
2024-09-01T00:00:00.000Z,31
2024-10-01T00:00:00.000Z,45


Databricks visualization. Run in Databricks to view.

In [0]:
-- Bar Chart - Cohort Size (2024 & 2025)

SELECT *
FROM workspace.ubiratan_cohort_project.cohort_customers
WHERE cohort_month >= DATE '2024-01-01'
  AND cohort_month <  DATE '2026-01-01';

customer_id,first_purchase_date,cohort_month,second_purchase_date,days_to_second_purchase
CUST272,2024-05-17,2024-05-01T00:00:00.000Z,2025-09-05,476
CUST522,2024-01-02,2024-01-01T00:00:00.000Z,2025-01-25,389
CUST506,2024-12-23,2024-12-01T00:00:00.000Z,null,null
CUST235,2024-06-11,2024-06-01T00:00:00.000Z,2025-03-05,267
CUST333,2024-07-17,2024-07-01T00:00:00.000Z,2025-03-01,227
CUST321,2024-12-21,2024-12-01T00:00:00.000Z,null,null
CUST223,2024-05-30,2024-05-01T00:00:00.000Z,2025-02-15,261
CUST337,2024-08-24,2024-08-01T00:00:00.000Z,null,null
CUST134,2024-10-26,2024-10-01T00:00:00.000Z,null,null
CUST580,2024-01-06,2024-01-01T00:00:00.000Z,2025-05-22,502


Databricks visualization. Run in Databricks to view.

In [0]:
-- Line Chart - Retention to 2nd Purchase (2024 Only)

SELECT *
FROM workspace.ubiratan_cohort_project.cohort_customers
WHERE cohort_month >= DATE '2024-01-01'
  AND cohort_month <  DATE '2025-01-01'
  AND days_to_second_purchase IS NOT NULL;

customer_id,first_purchase_date,cohort_month,second_purchase_date,days_to_second_purchase
CUST272,2024-05-17,2024-05-01T00:00:00.000Z,2025-09-05,476
CUST522,2024-01-02,2024-01-01T00:00:00.000Z,2025-01-25,389
CUST235,2024-06-11,2024-06-01T00:00:00.000Z,2025-03-05,267
CUST333,2024-07-17,2024-07-01T00:00:00.000Z,2025-03-01,227
CUST223,2024-05-30,2024-05-01T00:00:00.000Z,2025-02-15,261
CUST580,2024-01-06,2024-01-01T00:00:00.000Z,2025-05-22,502
CUST338,2024-03-28,2024-03-01T00:00:00.000Z,2025-04-27,395
CUST502,2024-01-14,2024-01-01T00:00:00.000Z,2025-04-16,458
CUST300,2024-04-20,2024-04-01T00:00:00.000Z,2025-01-05,260
CUST203,2024-01-19,2024-01-01T00:00:00.000Z,2025-03-14,420


Databricks visualization. Run in Databricks to view.

In [0]:
-- Line Chart - Retention to 2nd Purchase (2024 & 2025)

SELECT *
FROM workspace.ubiratan_cohort_project.cohort_customers
WHERE cohort_month >= DATE '2024-01-01'
  AND cohort_month <  DATE '2026-01-01'
  AND days_to_second_purchase IS NOT NULL;

customer_id,first_purchase_date,cohort_month,second_purchase_date,days_to_second_purchase
CUST272,2024-05-17,2024-05-01T00:00:00.000Z,2025-09-05,476
CUST522,2024-01-02,2024-01-01T00:00:00.000Z,2025-01-25,389
CUST235,2024-06-11,2024-06-01T00:00:00.000Z,2025-03-05,267
CUST333,2024-07-17,2024-07-01T00:00:00.000Z,2025-03-01,227
CUST223,2024-05-30,2024-05-01T00:00:00.000Z,2025-02-15,261
CUST580,2024-01-06,2024-01-01T00:00:00.000Z,2025-05-22,502
CUST338,2024-03-28,2024-03-01T00:00:00.000Z,2025-04-27,395
CUST502,2024-01-14,2024-01-01T00:00:00.000Z,2025-04-16,458
CUST300,2024-04-20,2024-04-01T00:00:00.000Z,2025-01-05,260
CUST203,2024-01-19,2024-01-01T00:00:00.000Z,2025-03-14,420


Databricks visualization. Run in Databricks to view.

In [0]:
-- Pie Chart - Repeat Purchase Depth (2024 Only)

WITH customer_purchase_counts AS (
  SELECT
    customer_id,
    COUNT(order_id) AS total_orders
  FROM workspace.ubiratan_cohort_project.clean_orders
  WHERE order_date >= DATE '2024-01-01'
    AND order_date <  DATE '2025-01-01'
  GROUP BY customer_id
)
SELECT *
FROM customer_purchase_counts;

customer_id,total_orders
CUST272,1
CUST151,1
CUST554,1
CUST061,1
CUST578,1
CUST188,1
CUST670,1
CUST353,1
CUST039,1
CUST253,1


Databricks visualization. Run in Databricks to view.

In [0]:
-- Pie Chart - Repeat Purchase Depth (2024 & 2025)

WITH customer_purchase_counts AS (
  SELECT
    customer_id,
    COUNT(order_id) AS total_orders
  FROM workspace.ubiratan_cohort_project.clean_orders
  WHERE order_date >= DATE '2024-01-01'
    AND order_date <  DATE '2026-01-01'
  GROUP BY customer_id
)
SELECT *
FROM customer_purchase_counts;

customer_id,total_orders
CUST272,2
CUST151,4
CUST554,2
CUST061,1
CUST578,2
CUST188,2
CUST670,1
CUST353,2
CUST039,2
CUST253,1


Databricks visualization. Run in Databricks to view.